我想用金庸的这个数据集训练一个LoRA，来帮我写作金庸风格的短篇小说，如何做

原始数据（长篇小说原文）
    ↓
数据重构（短篇故事 SFT 格式）
    ↓
LoRA 训练（instruction-following）
    ↓
推理：给标题/设定 → 输出完整短篇

## Step 1: 数据重构（最关键）
### 策略A — 滑动窗口切片（快速）

In [ ]:
import re

def extract_chapters(text):
    """按章节切割，每章约 3000-8000 字"""
    # 金庸小说章节格式：第X回/章 标题
    pattern = r'第[零一二三四五六七八九十百千\d]+[回章节]\s*[^\n]*'
    chapters = re.split(pattern, text)
    titles = re.findall(pattern, text)
    
    result = []
    for title, content in zip(titles, chapters[1:]):
        if 500 < len(content) < 10000:  # 过滤异常章节
            result.append({"title": title.strip(), "content": content.strip()})
    return result

In [ ]:
# configs/short_story_qlora.yaml
model_name_or_path: Qwen/Qwen2.5-7B-Instruct
dataset: data/instructions/jinyong_short_story_sft.jsonl

# LoRA - 创作任务需要更大 rank
lora_r: 64          # 之前可能是 16/32，创作任务提升
lora_alpha: 128
lora_dropout: 0.05
target_modules:
  - q_proj
  - k_proj
  - v_proj
  - o_proj
  - gate_proj      # 加入 FFN 层，提升风格迁移
  - up_proj
  - down_proj

# 训练
num_train_epochs: 3
per_device_train_batch_size: 2
gradient_accumulation_steps: 8   # 等效 batch=16
learning_rate: 2e-4
lr_scheduler_type: cosine
warmup_ratio: 0.05

# 序列长度
max_seq_length: 2048   # 短篇 800-1500 字，留余量

# 量化
load_in_4bit: true
bnb_4bit_quant_type: nf4
bnb_4bit_compute_dtype: bfloat16

1. 增加 instruction 多样性
   - "写一个悲剧结局的武侠故事"
   - "描写一场酒馆中的高手相遇"
   - "以第一人称写一个江湖复仇故事"

2. 加入结构约束（在 instruction 中）
   - "按照：开场→冲突→高潮→结局 的结构写"

3. 数据混合
   - 70% 金庸原文片段
   - 30% 人工标注的高质量短篇 prompt-output 对